> **Repository execution note.** Start JupyterLab from the **repository root** and place authorised source data in that same root directory (see [README](../README.md) and [Data requirements](../data/README.md)). These notebooks were originally distributed as standalone packages: any older instructions referring to an included ZIP, a `test/` folder, or bundled data do not apply to this GitHub repository. The notebooks preserve their original input schemas and experiment configurations. Results cached in the original files are intentionally excluded from this public-ready copy.

# Notebook 08 — Confirmation: RealMLP, Spatial Profiles, and Stability

**Project:** Luís Silvano · *Knowledge-Based Systems* · 29 October 2026  
**Confirmatory question (retrospective):** Does the improvement of RealMLP-E over RealMLP-D persist when training is repeated with multiple random seeds? Do genuine clusters outperform a random control? How does performance compare with XGBoost-B?

Notebook 07 showed a preliminary improvement in RealMLP after adding spatial clusters. The aim is **not** to search for a new winning algorithm or tune hyperparameters against 2024: it is to assess **the stability of that difference**, using the same hotels, years, target data, metrics, and outlier policy.

**Unavoidable limitations:** 2021–2024 have already been explored; repeating random seeds does not turn these data into an independent test. The tourism points of interest may have been collected after the study period. Conclusions concern **retrospective prediction and representation**, never causal effects.


## 0. Instructions for the student

1. Keep this notebook in `test/` or one of its subfolders.
2. Retain the `resultados_espaciais_05/` folder generated by Notebook 05, including `09_indicadores_e_clusters_por_hotel.csv` and `10_painel_com_indicadores_espaciais.csv`.
3. Install dependencies in the **Jupyter kernel**: `%pip install numpy pandas scikit-learn matplotlib seaborn xgboost pytabkit torch`.
4. First run with `PERFIL = 'rapido'` (one year/one seed), then `PERFIL = 'principal'` (three years/five seeds).
5. The `k=4` and `k=6` sensitivity analysis and random-group control are **robustness tests**, not hyperparameters to be selected according to the most favourable results.

By default, the notebook requires complete `k=5` data. If these are missing, it raises an explicit error rather than silently falling back to `k=1`. Only `verificacao_k1` permits a technical check with the older data; it **cannot be used as scientific confirmation**.

**Outputs:** saved in `resultados_KBS08/`; the file `00_metadados_metodologia.json` records configuration choices and data provenance.


In [ ]:
from pathlib import Path
import os, json, time, hashlib, warnings, importlib.util, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             adjusted_rand_score)

ROOT = Path.cwd()
OUT = ROOT / 'resultados_KBS08'
OUT.mkdir(parents=True, exist_ok=True)

# Change only this option initially: 'rapido', 'principal', 'publicacao', 'verificacao_k1'
PERFIL = os.getenv('KBS08_PERFIL', 'principal')
CONFIG = {
    'rapido': {'anos': [2023], 'sementes': [42], 'sensibilidade': False, 'placebo': False},
    'principal': {'anos': [2021, 2022, 2023], 'sementes': [11, 23, 42, 71, 101], 'sensibilidade': True, 'placebo': True},
    'publicacao': {'anos': [2021, 2022, 2023], 'sementes': [11, 23, 42, 71, 101, 137, 163, 197, 227, 263], 'sensibilidade': True, 'placebo': True},
    'verificacao_k1': {'anos': [2023], 'sementes': [42], 'sensibilidade': False, 'placebo': False},
}
if PERFIL not in CONFIG: raise ValueError('PERFIL desconhecido: '+PERFIL)
OP = CONFIG[PERFIL]
ANOS_CV = OP['anos']
SEMENTES = OP['sementes']
K_PRINCIPAL = 5
K_ALTERNATIVOS = [4, 6]
# Alternative analyses use 2023, without searching for the best k
ANOS_SENSIBILIDADE = [2023]
SEMENTES_SENSIBILIDADE = [s for s in SEMENTES if s in (11, 42, 71)] or SEMENTES[:1]
N_BOOT = 1000 if PERFIL in ('principal', 'publicacao') else 100
PERCENTIL_ALVO = 0.995
SEED_CLUSTER = 42  # fixed for the primary experiments
THREADS = 2
# 2024 was examined in earlier studies and is therefore disabled by default.
AVALIAR_2024_RETROSPETIVO = False
EXIGIR_K5 = PERFIL != 'verificacao_k1'
plt.rcParams.update({'figure.dpi':110, 'savefig.dpi':180, 'font.size':10})

# The package uses the same data-treatment policy as Notebook 07.
print('Profile:', PERFIL, '| validation years:', ANOS_CV, '| seeds:', SEMENTES)
print('Data folder:', ROOT.resolve())
print('k sensitivity:', OP['sensibilidade'], '| placebo:', OP['placebo'])

## 1. Data loading and file audit

The hotel join key is `ht_registration_number` in the panel and `hotel_key` in the spatial dataset. The panel may be stored in the consolidated CSV (Notebook 05) or the originally exported subset.

> Cluster labels from Notebook 05 are not used as predictive inputs. The clusters in E are **re-estimated from scratch within every fold** using spatial indices of training hotels, without using the target variable.


In [ ]:
def select_existing_file(candidatos, descricao):
    encontrado = next((p for p in candidatos if p.is_file()), None)
    if encontrado is None:
        raise FileNotFoundError(f'{descricao} not found. Checked locations: ' + ' | '.join(str(x) for x in candidatos))
    return encontrado

def normalize_id(v):
    if pd.isna(v): return None
    try:
        n = float(v)
        if np.isfinite(n) and n.is_integer(): return str(int(n))
    except (TypeError, ValueError): pass
    return str(v).strip() or None

BASES=[ROOT,ROOT.parent,ROOT/'KBS07_Benchmark_Integrado',ROOT.parent/'KBS07_Benchmark_Integrado']
CAND_PAINEL = ([b/'resultados_espaciais_05'/'10_painel_com_indicadores_espaciais.csv' for b in BASES]
             +[b/'10_painel_com_indicadores_espaciais.csv' for b in BASES]
             +[b/'hoteis_2018_2024_subset.csv' for b in BASES])
CAND_ESPACO = ([b/'resultados_espaciais_05'/'09_indicadores_e_clusters_por_hotel.csv' for b in BASES]
             +[b/'09_indicadores_e_clusters_por_hotel.csv' for b in BASES]
             +[b/'indicadores_espaciais_05.csv' for b in BASES])
ARQ_PAINEL=select_existing_file(CAND_PAINEL,'Annual panel')
ARQ_ESPACO=select_existing_file(CAND_ESPACO,'Spatial indicators')
painel=pd.read_csv(ARQ_PAINEL,low_memory=False)
espaco=pd.read_csv(ARQ_ESPACO,low_memory=False)

REQ = ['f_year','f_labprodct','ht_hotel_chain_d','ht_capacity','ht_quality_stars','ht_nuts_2_newcriteria']
faltam=[c for c in REQ if c not in painel.columns]
if faltam: raise ValueError(f'Required columns are missing from the panel: {faltam}')
if 'hotel_key' not in espaco: raise ValueError('hotel_key is missing from the spatial file')
if 'hotel_key' not in painel:
    if 'ht_registration_number' not in painel: raise ValueError('hotel_key and ht_registration_number are missing from the panel')
    painel['hotel_key']=painel.ht_registration_number.map(normalize_id)
else: painel['hotel_key']=painel.hotel_key.map(normalize_id)
espaco['hotel_key']=espaco.hotel_key.map(normalize_id)
if espaco.hotel_key.isna().any() or espaco.hotel_key.duplicated().any():
    raise ValueError('hotel_key is missing or duplicated in the spatial file')

K5=[f'a_{x}_k5' for x in ('castelos','igrejas','praias')]
K1=[f'a_{x}_k1' for x in ('castelos','igrejas','praias')]
completo_k5=all(c in espaco and pd.to_numeric(espaco[c],errors='coerce').notna().mean()>=.95 for c in K5)
if EXIGIR_K5 and not completo_k5:
    cobertura={c: round(pd.to_numeric(espaco[c],errors='coerce').notna().mean(),3) if c in espaco else 0 for c in K5}
    raise ValueError('This experiment requires complete k=5 indicators from Notebook 05. '
                     f'Current coverage: {cobertura}. Place the updated 09_ file in resultados_espaciais_05/.')
K_ESPACO=5 if completo_k5 else 1
SP_COLS=[f'a_{x}_k{K_ESPACO}' for x in ('castelos','igrejas','praias')]+['log_dens_restaurantes_r0p8km']
for c in SP_COLS:
    if c not in espaco: raise ValueError('Missing indicator: '+c)
    espaco[c]=pd.to_numeric(espaco[c],errors='coerce')

# Keep spatial columns exclusively in table 09 (prevents _x/_y suffixes and duplicates).
COLS_ANTIGAS=[c for c in painel if c.startswith(('a_castelos_k','a_igrejas_k','a_praias_k',
                    'log_dens_restaurantes_','dens_restaurantes_','n_restaurantes_r'))]
painel=painel.drop(columns=COLS_ANTIGAS+['cluster','perfil_nome'],errors='ignore')
for c in ['f_year','f_labprodct','ht_hotel_chain_d']:
    painel[c]=pd.to_numeric(painel[c],errors='coerce')
painel=painel[painel.f_year.between(2018,2024)&painel.ht_hotel_chain_d.eq(0)&painel.hotel_key.notna()].copy()
painel['f_year']=painel['f_year'].astype(int)
if painel.duplicated(['hotel_key','f_year']).any(): raise ValueError('Duplicate hotel–year pairs in panel')
base=painel.merge(espaco[['hotel_key']+SP_COLS],on='hotel_key',how='inner',validate='many_to_one')
base=base.replace([np.inf,-np.inf],np.nan)
base=base[base[SP_COLS].notna().sum(axis=1)>=3].copy()
print('Panel:',ARQ_PAINEL)
print('Indicators:',ARQ_ESPACO)
print('spatial k:',K_ESPACO, '| independent hotels:',base.hotel_key.nunique(),'| rows:',len(base))
print('k=5 coverage:', {c:round(pd.to_numeric(espaco[c],errors='coerce').notna().mean(),3) if c in espaco else 0 for c in K5})

## 2. Hotel–year pairs and temporal policy

We predict productivity in year \(t\) from hotel structure and region observed in the preceding year and, for models D/E, productivity in \(t-1\). Only consecutive-year pairs are accepted:

$ \hat{y}_{i,t} = f(\mathbf{s}_{i,t-1},\; \mathbf{a}_i,\; y_{i,t-1},\; t). $

**XGBoost-B** does **not** receive \(y_{i,t-1}\) and serves as a strong structure-plus-space benchmark. RealMLP-D receives historical productivity and continuous spatial indicators; E additionally receives one-hot spatial-group labels.

Imputation statistics, normalisation, outlier handling, and clustering are computed **exclusively from the training portion of each fold**. D and E are always compared on the same sample.


In [ ]:
ESTRUTURA=['ht_capacity','ht_quality_stars','ht_number_of_accommodation_units',
           'ht_spa_d','ht_gym_d','ht_outdoor_pools_d','ht_meeting_rooms_d']
for col in ESTRUTURA:
    if col not in base: base[col]=np.nan
    base[col]=pd.to_numeric(base[col],errors='coerce')

prev=base[['hotel_key','f_year','f_labprodct','ht_nuts_2_newcriteria']+ESTRUTURA].copy()
prev=prev.rename(columns={'f_labprodct':'lag_y','ht_nuts_2_newcriteria':'regiao_lag',
                      **{c:'lag_'+c for c in ESTRUTURA}})
prev['f_year']=prev.f_year+1
atual=base[['hotel_key','f_year','f_labprodct']+SP_COLS].copy()
PARES=atual.merge(prev,on=['hotel_key','f_year'],how='inner',validate='one_to_one')
PARES=PARES.rename(columns={'f_labprodct':'y'}).replace([np.inf,-np.inf],np.nan)
PARES=PARES.dropna(subset=['y','lag_y']).copy()
PARES=PARES[PARES.f_year.between(2019,2024)].sort_values(['f_year','hotel_key']).reset_index(drop=True)
PARES['regiao_lag']=PARES.regiao_lag.fillna('Region not specified').astype(str)
PARES['ano_previsto']=PARES.f_year.astype(float)
if PARES.duplicated(['hotel_key','f_year']).any(): raise ValueError('Duplicate hotel–year pairs')
if PARES.empty: raise ValueError('No valid hotel–year pairs are available for the study')
assert PARES.y.notna().all() and PARES.lag_y.notna().all()
AUDIT=PARES.groupby('f_year').agg(hoteis=('hotel_key','nunique'),amostras=('y','size'),
                                  produtividade_mediana=('y','median')).reset_index()
AUDIT.to_csv(OUT/'01_auditoria_amostra.csv',index=False)
display(AUDIT.round(2))

## 3. Three models and the fold-specific spatial representation

The comparison fixes the parameters used in Notebook 07:

- **RealMLP-D:** structure, region, year, previous productivity, and four spatial indicators.
- **RealMLP-E5:** the same features + one-hot encoding of five spatial clusters learned from training data.
- **XGBoost-B:** structure, region, year, and four spatial indicators; no previous productivity.

The primary analysis always uses five clusters. Alternative values \(k\in\{4,6\}\) serve only to assess stability. The **E5-placebo** control replaces genuine clusters with pseudorandom labels that remain constant for each hotel, preserving the number of columns without using geography or the target.

**Accessibility indices:** \(A_i^{(c)}(5)=\frac15\sum_{j=1}^{5}(1+d_{ij}^{(c)}/1\mathrm{km})^{-1}\). Group membership is derived from these previously computed dimensions, not from productivity.


In [ ]:
BASE_NUM=['lag_'+x for x in ESTRUTURA]+['ano_previsto']
FEATS={'B':BASE_NUM+SP_COLS,
       'D':BASE_NUM+['lag_y']+SP_COLS,
       'E':BASE_NUM+['lag_y']+SP_COLS}

def fit_clusters_on_training_data(train, valid, k, modo='geografico'):
    if modo=='placebo':
        def assign_cluster_label(hotel):
            hx=hashlib.sha256(('KBS08-placebo-v1:'+str(hotel)).encode('utf-8')).digest()
            return int.from_bytes(hx[:8],'little') % k
        return (np.array([assign_cluster_label(v) for v in train.hotel_key],dtype=int),
                np.array([assign_cluster_label(v) for v in valid.hotel_key],dtype=int))
    unicos=train.drop_duplicates('hotel_key')
    imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(unicos[SP_COLS])
    arr=imp.transform(unicos[SP_COLS]); lo=np.quantile(arr,.01,axis=0); hi=np.quantile(arr,.99,axis=0)
    scale=StandardScaler().fit(np.clip(arr,lo,hi))
    km=KMeans(n_clusters=k,n_init=12,random_state=SEED_CLUSTER)
    km.fit(scale.transform(np.clip(arr,lo,hi)))
    def assign_cluster_labels(frame):
        a=imp.transform(frame[SP_COLS]); a=scale.transform(np.clip(a,lo,hi))
        return km.predict(a)
    return assign_cluster_labels(train),assign_cluster_labels(valid)

def prepare_fold_matrices(train,valid,cenario='D',k=5,modo='geografico'):
    cols=FEATS[cenario]
    tr=train[cols].astype(float).copy().replace([np.inf,-np.inf],np.nan)
    va=valid[cols].astype(float).copy().replace([np.inf,-np.inf],np.nan)
    if 'lag_y' in cols:
        low,high=np.quantile(train.lag_y,[.005,.995])
        tr['lag_y']=tr['lag_y'].clip(low,high);va['lag_y']=va['lag_y'].clip(low,high)
    imp=SimpleImputer(strategy='median',keep_empty_features=True)
    a=imp.fit_transform(tr); b=imp.transform(va)
    scaler=StandardScaler().fit(a)
    a_s=scaler.transform(a); b_s=scaler.transform(b)
    try: ohe=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
    except TypeError: ohe=OneHotEncoder(handle_unknown='ignore',sparse=False)
    one_tr=ohe.fit_transform(train[['regiao_lag']].astype(str))
    one_va=ohe.transform(valid[['regiao_lag']].astype(str))
    extras_tr=[];extras_va=[]
    if cenario=='E':
        groups_tr,groups_va=fit_clusters_on_training_data(train,valid,k,modo)
        extras_tr=[np.eye(k)[groups_tr]];extras_va=[np.eye(k)[groups_va]]
    raw_tr=np.column_stack([a,one_tr]+extras_tr).astype('float32',copy=True)
    raw_va=np.column_stack([b,one_va]+extras_va).astype('float32',copy=True)
    scale_tr=np.column_stack([a_s,one_tr]+extras_tr).astype('float32',copy=True)
    scale_va=np.column_stack([b_s,one_va]+extras_va).astype('float32',copy=True)
    assert raw_tr.shape[0]==len(train) and scale_va.shape[0]==len(valid)
    assert np.isfinite(scale_tr).all() and np.isfinite(scale_va).all()
    return {'raw_tr':raw_tr,'raw_va':raw_va,'scale_tr':scale_tr,'scale_va':scale_va,
            'cols':cols}

def build_model(nome,seed):
    if nome=='xgboost':
        from xgboost import XGBRegressor
        return XGBRegressor(n_estimators=280,max_depth=4,learning_rate=.045,
              subsample=.9,colsample_bytree=.9,objective='reg:squarederror',
              tree_method='hist',n_jobs=THREADS,random_state=int(seed))
    if nome=='realmlp':
        import torch
        from pytabkit import RealMLP_TD_Regressor
        torch.manual_seed(int(seed)); np.random.seed(int(seed))
        torch.set_num_threads(THREADS)
        dev='cuda' if torch.cuda.is_available() else 'cpu'
        return RealMLP_TD_Regressor(device=dev,random_state=int(seed))
    raise ValueError('Unrecognised model: '+nome)

def train_model(nome,train,valid,mat,seed):
    modelo=build_model(nome,seed)
    if nome=='realmlp': xtr=mat['scale_tr'];xva=mat['scale_va']
    else: xtr=mat['raw_tr'];xva=mat['raw_va']
    ytr=train.y.to_numpy(dtype=float,copy=True)
    teto=float(np.quantile(ytr,PERCENTIL_ALVO))
    ytr=np.ascontiguousarray(np.minimum(ytr,teto),dtype=float)
    inicio=time.perf_counter()
    modelo.fit(np.array(xtr,copy=True),ytr)
    pred=np.asarray(modelo.predict(np.array(xva,copy=True)),dtype=float).reshape(-1)
    tempo=time.perf_counter()-inicio
    if len(pred)!=len(valid) or not np.isfinite(pred).all():
        raise RuntimeError('Model produced invalid predictions')
    return pred,tempo,teto

def rmse(y,p): return float(np.sqrt(mean_squared_error(y,p)))
def compute_metrics(y,p):
    return {'MAE':float(mean_absolute_error(y,p)),
            'RMSE':rmse(y,p), 'R2':float(r2_score(y,p))}

# Check the installation before starting long training runs.
if PERFIL!='verificacao_k1' and importlib.util.find_spec('pytabkit') is None:
    raise ImportError('RealMLP is not installed in this kernel. Run: %pip install pytabkit')
print('RealMLP interface:', 'pytabkit installed' if importlib.util.find_spec('pytabkit') else 'not installed (QA mode)')

## 4. Main experiment — paired repetitions

For each year \(t\), we fit all three algorithms to the same hotels, using training observations whose target years precede \(t\). Each seed is used for both RealMLP-D and RealMLP-E, creating paired comparisons.

$
\Delta\mathrm{MAE}_{E-D}=\frac1N\sum_i \left(|y_i-\hat y_{i,E}|-|y_i-\hat y_{i,D}|\right).
$

**Negative values mean E improves upon D.** We report MAE, RMSE, \(R^2\), RMSE relative to the training mean, and runtimes separately. Models remain fixed, without hyperparameter changes to search for better results.


In [ ]:
LINHAS=[]; PREDICOES=[]; FALHAS=[]
ANOS_EXEC=ANOS_CV + ([2024] if AVALIAR_2024_RETROSPETIVO else [])
for ano in ANOS_EXEC:
    train=PARES[PARES.f_year<ano].copy()
    valid=PARES[PARES.f_year==ano].copy()
    assert not train.empty and not valid.empty and train.f_year.max()<ano
    rmse_baseline=rmse(valid.y,np.repeat(train.y.mean(),len(valid)))
    ms={'B':prepare_fold_matrices(train,valid,'B'),
        'D':prepare_fold_matrices(train,valid,'D'),
        'E':prepare_fold_matrices(train,valid,'E',k=K_PRINCIPAL)}
    pares=[('xgboost','B')]
    if PERFIL!='verificacao_k1': pares += [('realmlp','D'),('realmlp','E')]
    print(f'\nYear {ano}: {len(train)} training / {len(valid)} validation; '+
          f'{len(SEMENTES)} seeds and {len(pares)} combinations')
    for seed in SEMENTES:
        for nome,cenario in pares:
            try:
                pred,seg,teto=train_model(nome,train,valid,ms[cenario],seed)
                score=compute_metrics(valid.y,pred)
                LINHAS.append({'ano':ano,'semente':seed,'modelo':nome,'cenario':cenario,
                               'n_treino':len(train),'n_validacao':len(valid),
                               'RMSE_rel_dummy':score['RMSE']/rmse_baseline,
                               'segundos':seg,'teto_alvo_treino':teto,**score})
                PREDICOES.append(pd.DataFrame({'hotel_key':valid.hotel_key.to_numpy(),
                     'ano':ano,'semente':seed,'modelo':nome,'cenario':cenario,
                     'real':valid.y.to_numpy(dtype=float),'predito':pred}))
            except Exception as exc:
                FALHAS.append({'ano':ano,'semente':seed,'modelo':nome,'cenario':cenario,
                               'erro_tipo':type(exc).__name__,'erro':str(exc)[:400]})
                print('FAILED:',ano,seed,nome,cenario,str(exc)[:180])
    print('Training runs completed for year:',ano, '->',len([x for x in LINHAS if x['ano']==ano]))

METRICAS=pd.DataFrame(LINHAS)
if METRICAS.empty: raise RuntimeError('No model was executed')
PREDS=pd.concat(PREDICOES,ignore_index=True)
METRICAS.to_csv(OUT/'02_metricas_por_ano_semente.csv',index=False)
PREDS.to_csv(OUT/'03_previsoes_individuais.csv',index=False)
pd.DataFrame(FALHAS,columns=['ano','semente','modelo','cenario','erro_tipo','erro']).to_csv(
    OUT/'04_falhas_execucao.csv',index=False)

REF=[]
for ano in ANOS_EXEC:
    v=PARES[PARES.f_year==ano]
    REF.append({'ano':ano,'modelo':'persistencia','cenario':'P','n':len(v),
                **compute_metrics(v.y,v.lag_y.to_numpy(float))})
REF_DF=pd.DataFrame(REF)
REF_DF.to_csv(OUT/'02b_persistencia_referencia.csv',index=False)
print('Simple temporal baseline (y[t−1]):')
display(REF_DF.round(3))

print('Summary (mean across seeds for each year):')
display(METRICAS.groupby(['modelo','cenario','ano'],as_index=False).agg(
   MAE=('MAE','mean'),RMSE=('RMSE','mean'),R2=('R2','mean'),
   RMSE_rel_dummy=('RMSE_rel_dummy','mean'),desvio_MAE=('MAE','std')).round(3))

## 5. Paired D vs E comparison and uncertainty

The same hotel–year observation is evaluated by both models with the same seed. We calculate individual error differences and a **hotel-level bootstrap** for the mean difference (grouping temporal repetitions). Before bootstrapping, we aggregate errors across seeds so that five model fits are not incorrectly treated as five independent hotel samples.

These intervals describe variation among hotels **conditional on the years and previously selected configurations**. They do not provide external confirmation or account fully for dependence among hotels due to nationwide shocks.


In [ ]:
RESULT_D=[]; DIFF_PARES=[]
for ano in ANOS_EXEC:
    for seed in SEMENTES:
        l=PREDS[(PREDS.ano==ano)&(PREDS.semente==seed)&(PREDS.modelo=='realmlp')&(PREDS.cenario=='D')]
        r=PREDS[(PREDS.ano==ano)&(PREDS.semente==seed)&(PREDS.modelo=='realmlp')&(PREDS.cenario=='E')]
        if l.empty or r.empty: continue
        j=l.merge(r,on=['hotel_key','ano','semente'],suffixes=('_D','_E'),validate='one_to_one')
        assert len(j)==len(l)==len(r) and np.allclose(j.real_D,j.real_E)
        j['ae_D']=np.abs(j.real_D-j.predito_D);j['ae_E']=np.abs(j.real_E-j.predito_E)
        j['se_D']=(j.real_D-j.predito_D)**2;j['se_E']=(j.real_E-j.predito_E)**2
        j['delta_AE']=j.ae_E-j.ae_D
        DIFF_PARES.append(j[['hotel_key','ano','semente','ae_D','ae_E','se_D','se_E','delta_AE']])
        RESULT_D.append({'ano':ano,'semente':seed,'n':len(j),
            'delta_MAE_E_menos_D':float(j.delta_AE.mean()),
            'delta_RMSE_E_menos_D':float(np.sqrt(j.se_E.mean())-np.sqrt(j.se_D.mean())),
            'percentagem_hoteis_E_melhor':100*float((j.ae_E<j.ae_D).mean())})
PARES_DF=pd.concat(DIFF_PARES,ignore_index=True) if DIFF_PARES else pd.DataFrame(columns=['hotel_key','ano','semente','ae_D','ae_E','se_D','se_E','delta_AE'])
DELTA_SEEDS=pd.DataFrame(RESULT_D,columns=['ano','semente','n','delta_MAE_E_menos_D','delta_RMSE_E_menos_D','percentagem_hoteis_E_melhor'])
PARES_DF.to_csv(OUT/'05_erros_pareados_D_E.csv',index=False)
DELTA_SEEDS.to_csv(OUT/'06_diferencas_por_ano_semente.csv',index=False)
if not DELTA_SEEDS.empty:
    print('Differences by year and seed — negative favours RealMLP-E:')
    display(DELTA_SEEDS.round(3))
    print('Share of runs in which E outperformed D on MAE:',
          round(float((DELTA_SEEDS.delta_MAE_E_menos_D<0).mean()),3))
else:
    print('No RealMLP D/E pairs found (check profile, installation, or failures).')

BOOT=[]
if not PARES_DF.empty:
    # Each hotel–year contributes once after averaging across seeds.
    agregado=PARES_DF.groupby(['hotel_key','ano'],as_index=False)[['ae_D','ae_E','se_D','se_E']].mean()
    rng=np.random.default_rng(20261029)
    for etiqueta,df in [('agregado_CV',agregado)]+[(str(y),agregado[agregado.ano==y]) for y in ANOS_CV]:
        if df.empty: continue
        grupo=df.groupby('hotel_key',sort=False).agg(
            n=('ae_D','size'),sum_d=('ae_D','sum'),sum_e=('ae_E','sum'),
            sq_d=('se_D','sum'),sq_e=('se_E','sum'))
        g=grupo.to_numpy(dtype=float); N=len(g)
        vals=np.zeros((N_BOOT,2),dtype=float)
        for b in range(N_BOOT):
            ids=rng.integers(0,N,size=N)
            q=g[ids].sum(axis=0)
            n,smd,sme,sqd,sqe=q
            vals[b]=[(sme-smd)/n, np.sqrt(sqe/n)-np.sqrt(sqd/n)]
        n,smd,sme,sqd,sqe=g.sum(axis=0)
        BOOT.append({'periodo':etiqueta,'hoteis':N,'hotel_anos':int(n),
            'delta_MAE':(sme-smd)/n,'IC95_MAE_inf':np.quantile(vals[:,0],.025),
            'IC95_MAE_sup':np.quantile(vals[:,0],.975),
            'delta_RMSE':np.sqrt(sqe/n)-np.sqrt(sqd/n),
            'IC95_RMSE_inf':np.quantile(vals[:,1],.025),
            'IC95_RMSE_sup':np.quantile(vals[:,1],.975)})
BOOT_DF=pd.DataFrame(BOOT,columns=['periodo','hoteis','hotel_anos','delta_MAE','IC95_MAE_inf','IC95_MAE_sup','delta_RMSE','IC95_RMSE_inf','IC95_RMSE_sup'])
BOOT_DF.to_csv(OUT/'07_bootstrap_pareado_por_hotel.csv',index=False)
if not BOOT_DF.empty: display(BOOT_DF.round(3))

## 6. Robustness to cluster definitions and placebo control

The additional feature in E represents **a partition of spatial indicators that are already present**. It therefore contains no independent information beyond the spatial context. We ask whether *this representation of the information* helps RealMLP.

- **E4 / E6:** assess the sensitivity of results to the number of groups only.
- **E5-placebo:** retain the same number of binary dimensions, but use pseudorandom hotel-stable groups unrelated to tourist points of interest.

The placebo is an **additional representation control**, not causal evidence. Because a random group remains stable for each hotel, it may encode unobserved hotel-level heterogeneity; interpret its performance with that limitation. Evaluation is restricted to 2023 and three prespecified seeds, to avoid turning this stage into another optimisation exercise.


In [ ]:
SENS=[]; PR_PLACEBO=[]
if OP['sensibilidade'] and not PARES_DF.empty:
    for ano in ANOS_SENSIBILIDADE:
        tr=PARES[PARES.f_year<ano].copy();va=PARES[PARES.f_year==ano].copy()
        for seed in SEMENTES_SENSIBILIDADE:
            for k in K_ALTERNATIVOS:
                try:
                    mat=prepare_fold_matrices(tr,va,'E',k=k)
                    pred,secs,_=train_model('realmlp',tr,va,mat,seed)
                    score=compute_metrics(va.y,pred)
                    SENS.append({'ano':ano,'semente':seed,'modelo':'realmlp',
                         'cenario':f'E{k}', 'k_clusters':k,'segundos':secs,**score})
                except Exception as e:
                    FALHAS.append({'ano':ano,'semente':seed,'modelo':'realmlp',
                      'cenario':f'E{k}','erro_tipo':type(e).__name__,'erro':str(e)[:400]})
if OP['placebo'] and not PARES_DF.empty:
    for ano in ANOS_SENSIBILIDADE:
        tr=PARES[PARES.f_year<ano].copy();va=PARES[PARES.f_year==ano].copy()
        mat=prepare_fold_matrices(tr,va,'E',k=K_PRINCIPAL,modo='placebo')
        for seed in SEMENTES_SENSIBILIDADE:
            try:
                pred,secs,_=train_model('realmlp',tr,va,mat,seed)
                score=compute_metrics(va.y,pred)
                SENS.append({'ano':ano,'semente':seed,'modelo':'realmlp',
                    'cenario':'E5_placebo','k_clusters':K_PRINCIPAL,'segundos':secs,**score})
                PR_PLACEBO.append(pd.DataFrame({'hotel_key':va.hotel_key.to_numpy(),
                    'ano':ano,'semente':seed,'real':va.y.to_numpy(dtype=float),'predito':pred}))
            except Exception as e:
                FALHAS.append({'ano':ano,'semente':seed,'modelo':'realmlp',
                    'cenario':'E5_placebo','erro_tipo':type(e).__name__,'erro':str(e)[:400]})
SENS_DF=pd.DataFrame(SENS,columns=['ano','semente','modelo','cenario','k_clusters',
    'segundos','MAE','RMSE','R2'])
SENS_DF.to_csv(OUT/'08_sensibilidade_clusters_placebo.csv',index=False)
if PR_PLACEBO:
    pd.concat(PR_PLACEBO,ignore_index=True).to_csv(OUT/'09_previsoes_placebo.csv',index=False)
if not SENS_DF.empty:
    principal=METRICAS[(METRICAS.modelo=='realmlp')&(METRICAS.ano.isin(ANOS_SENSIBILIDADE))&
                 (METRICAS.semente.isin(SEMENTES_SENSIBILIDADE))]
    basel=principal[principal.cenario.isin(['D','E'])].assign(
          cenario=lambda x:x.cenario.map({'D':'D','E':'E5'}))
    comparavel=pd.concat([basel[['cenario','ano','semente','MAE','RMSE','R2']],
                           SENS_DF[['cenario','ano','semente','MAE','RMSE','R2']]])
    print('Control/robustness (same seeds, year 2023):')
    display(comparavel.groupby('cenario')[['MAE','RMSE','R2']].agg(['mean','std']).round(3))
else:
    print('Sensibilidades desativadas neste perfil.')

## 7. Figures and final interpretation

Plots are generated **from actual executions**. No results are imported from Notebook 07. A negative \(\Delta\mathrm{MAE}=E-D\) favours spatial groups; a positive value does not.

Criteria for a cautious conclusion:

- D and E should show consistent improvements across seeds and years, rather than in a single run;
- the comparison with XGBoost-B and the persistence baseline should remain visible;
- genuine E5 must be compared against E5-placebo;
- k=4/6 serve only as sensitivity checks, not as a basis for selecting the most convenient k;
- an observed difference in 2021–2023 remains **retrospective evidence of algorithmic stability**, not new external evidence.


In [ ]:
# Mean annual results by model
if not METRICAS.empty:
    fig,ax=plt.subplots(figsize=(9.5,4.7))
    sns.lineplot(data=METRICAS.assign(combinacao=lambda d:d.modelo+'-'+d.cenario),
        x='ano',y='MAE',hue='combinacao',units='semente',estimator=None,
        alpha=.18,lw=.75,legend=False,ax=ax)
    resumo=METRICAS.groupby(['modelo','cenario','ano'],as_index=False)['MAE'].mean()
    sns.lineplot(data=resumo.assign(combinacao=lambda d:d.modelo+'-'+d.cenario),
        x='ano',y='MAE',hue='combinacao',marker='o',lw=2.4,ax=ax)
    ax.set(title='Prediction errors by year and seed',xlabel='Validation year',ylabel='MAE (original scale)')
    ax.plot(REF_DF.ano,REF_DF.MAE,linestyle='--',color='black',linewidth=1.7,marker='x',label='persistence')
    ax.legend(title='Model and scenario',loc='best');ax.grid(alpha=.16)
    plt.tight_layout();plt.savefig(OUT/'figura_01_mae_anos_sementes.png',bbox_inches='tight');plt.show()
if not DELTA_SEEDS.empty:
    fig,axs=plt.subplots(1,2,figsize=(11,4.2))
    sns.stripplot(data=DELTA_SEEDS,x='ano',y='delta_MAE_E_menos_D',jitter=.13,
                  alpha=.85,size=7,ax=axs[0]);axs[0].axhline(0,color='black',ls='--',lw=.8)
    axs[0].set(xlabel='Year',ylabel='ΔMAE: E − D',title='RealMLP: cluster effect by random seed')
    if not BOOT_DF.empty:
        sub=BOOT_DF[BOOT_DF.periodo.isin([str(x) for x in ANOS_CV])]
        if not sub.empty:
            yrs=sub.periodo.to_numpy()
            axs[1].errorbar(yrs,sub.delta_MAE,
                 yerr=np.array([sub.delta_MAE-sub.IC95_MAE_inf,sub.IC95_MAE_sup-sub.delta_MAE]),
                 fmt='o',capsize=4)
    axs[1].axhline(0,color='black',ls='--',lw=.8)
    axs[1].set(xlabel='Year',ylabel='ΔMAE: E − D',title='Hotel-level confidence intervals (conditional on years)')
    plt.tight_layout();plt.savefig(OUT/'figura_02_delta_realmlp_clusters.png',bbox_inches='tight');plt.show()
if not SENS_DF.empty:
    fig,ax=plt.subplots(figsize=(8,4.1))
    d=METRICAS[(METRICAS.modelo=='realmlp')&(METRICAS.ano.isin(ANOS_SENSIBILIDADE))&
              (METRICAS.semente.isin(SEMENTES_SENSIBILIDADE))]
    q=pd.concat([d[['cenario','MAE']].assign(cenario=lambda v:v.cenario.replace({'E':'E5'})),
                 SENS_DF[['cenario','MAE']]])
    sns.boxplot(data=q,x='cenario',y='MAE',color='#9EBDD1',ax=ax)
    sns.stripplot(data=q,x='cenario',y='MAE',color='black',size=4,alpha=.8,ax=ax)
    ax.set(title='Sensitivity to k and placebo control (2023)',xlabel='Representation',ylabel='MAE')
    plt.tight_layout();plt.savefig(OUT/'figura_03_sensibilidade_placebo.png',bbox_inches='tight');plt.show()

## 8. Final audit and recommendations for the manuscript

**What a positive result would justify writing:** “The categorical representation derived from spatial clusters yielded stable predictive gains across multiple RealMLP initialisations in the examined validation years, relative to the same architecture without this representation.”  
**What it would NOT justify:** claiming that location causes productivity, that RealMLP is superior under all conditions, or that these findings are equivalent to a blind external test.

This notebook does not choose the best k, does not change preprocessing based on evaluation results, and does not omit unsuccessful seeds. **Failures must be investigated before preparing the manuscript.**


In [ ]:
STATUS=[]
requeridas = [('xgboost','B')]+([] if PERFIL=='verificacao_k1' else [('realmlp','D'),('realmlp','E')])
for nome,c in requeridas:
    t=METRICAS[(METRICAS.modelo==nome)&(METRICAS.cenario==c)]
    STATUS.append({'modelo':nome,'cenario':c,'execucoes':len(t),
       'esperadas':len(ANOS_EXEC)*len(SEMENTES),
       'completo':len(t)==len(ANOS_EXEC)*len(SEMENTES)})
STATUS_DF=pd.DataFrame(STATUS)
STATUS_DF.to_csv(OUT/'10_status_execucao.csv',index=False)
if FALHAS:
    pd.DataFrame(FALHAS).to_csv(OUT/'04_falhas_execucao.csv',index=False)
metadata={
 'estudo':'KBS08 - estabilidade RealMLP-D/E e XGBoost-B','perfil':PERFIL,
 'painel_arquivo':str(ARQ_PAINEL.resolve()),'espaco_arquivo':str(ARQ_ESPACO.resolve()),
 'hash_sha256_painel':hashlib.sha256(ARQ_PAINEL.read_bytes()).hexdigest(),
 'hash_sha256_indicadores':hashlib.sha256(ARQ_ESPACO.read_bytes()).hexdigest(),
 'anos_cv':ANOS_CV,'sementes':SEMENTES,'n_pares_2019_2024':len(PARES),
 'k_indicadores':K_ESPACO,'colunas_espaciais':SP_COLS,
 'k_principal':K_PRINCIPAL,'k_alternativos':K_ALTERNATIVOS,
 'percentil_alvo_treino':PERCENTIL_ALVO,'seed_clusters':SEED_CLUSTER,
 'n_bootstrap':N_BOOT,'retrospetiva_2024_executada':AVALIAR_2024_RETROSPETIVO,
 'dependencias':{'python':platform.python_version(),'pandas':pd.__version__,
                 'sklearn':__import__('sklearn').__version__},
 'observacao':'Repeated initialisations measure algorithmic robustness, not a new external test.',
 'aviso_POIs':'Confirm when tourist attractions were surveyed; they may postdate 2024.'}
(OUT/'00_metadados_metodologia.json').write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding='utf-8')
print('Execution summary:');display(STATUS_DF)
print('Outputs in:',OUT.resolve())
print('Conclusion: read 07_bootstrap_pareado_por_hotel.csv together with 08_sensibilidade_clusters_placebo.csv and 04_falhas_execucao.csv')